# AffectLab calibrated fusion report

CPU-only reproducibility pass using the validation and test predictions already stored in private Cloud Storage. Produces auditable calibrated predictions, NLL, multiclass Brier score, ECE, and reliability bins.

In [ ]:
import base64, json, os, subprocess, sys
from pathlib import Path
from google.colab import auth, userdata
PROJECT_ID, BUCKET = 'cat-behaviour-research', 'affectlab-research-raluca-biras'
TEXT_EXPERIMENT = 'iemocap_benchmark4_context3_deberta_v3_small'
AUDIO_EXPERIMENT = 'iemocap_benchmark4_audio_wav2vec2_base'
auth.authenticate_user()
subprocess.run(['gcloud', 'config', 'set', 'project', PROJECT_ID], check=True)
repo = Path('/content/emotion-aware-role-play-model')
token = userdata.get('GITHUB_TOKEN')
if not token: raise RuntimeError('Add GITHUB_TOKEN to Colab Secrets.')
header = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
option = f'http.extraHeader=Authorization: Basic {header}'
url = 'https://github.com/ralucabiras/emotion-aware-role-play-model.git'
if not repo.exists(): subprocess.run(['git', '-c', option, 'clone', url, str(repo)], check=True)
else: subprocess.run(['git', '-C', str(repo), '-c', option, 'pull', '--ff-only'], check=True)
del token, header, option
os.chdir(repo)

In [ ]:
work = Path('/content/calibration-report')
for fold in range(1, 6):
    for family, experiment, modality in (('iemocap-text', TEXT_EXPERIMENT, 'text'), ('iemocap-audio', AUDIO_EXPERIMENT, 'audio')):
        destination = work / modality / f'fold-{fold}'
        destination.mkdir(parents=True, exist_ok=True)
        remote = f'gs://{BUCKET}/runs/{family}/{experiment}/fold-{fold}'
        for filename in ('metrics.json', 'test_predictions.jsonl', 'validation_predictions.jsonl'):
            subprocess.run(['gcloud', 'storage', 'cp', f'{remote}/{filename}', str(destination/filename)], check=True)

In [ ]:
output = work / 'output'
subprocess.run([sys.executable, '-m', 'ml.evaluation.calibrate_iemocap_fusion', '--text-dir', str(work/'text'), '--audio-dir', str(work/'audio'), '--output-dir', str(output)], check=True)
result = json.loads((output/'summary.json').read_text())
display({'fold_parameters': [{'fold': x['fold'], 'text_weight': x['text_weight'], 'temperature': x['temperature']} for x in result['folds']], 'pooled': result['pooled']})
global_output = work / 'global-oof-output'
subprocess.run([sys.executable, '-m', 'ml.evaluation.calibrate_iemocap_fusion', '--text-dir', str(work/'text'), '--audio-dir', str(work/'audio'), '--output-dir', str(global_output), '--scope', 'global-oof'], check=True)
global_result = json.loads((global_output/'summary.json').read_text())
display({'global_parameters': {'text_weight': global_result['text_weight'], 'audio_weight': global_result['audio_weight'], 'temperature': global_result['temperature'], 'validation_examples': global_result['validation_examples']}, 'pooled': global_result['pooled']})

In [ ]:
remote = f'gs://{BUCKET}/runs/iemocap-fusion/iemocap_benchmark4_validation_fitted_fusion'
subprocess.run(['gcloud', 'storage', 'rsync', '--recursive', str(output), remote], check=True)
global_remote = f'gs://{BUCKET}/runs/iemocap-fusion/iemocap_benchmark4_global_oof_calibrated_fusion'
subprocess.run(['gcloud', 'storage', 'rsync', '--recursive', str(global_output), global_remote], check=True)
print('Updated private calibration reports:', remote, global_remote)